# BERTurk fine-tune + ONNX INT8 quantization (Colab, GPU)

Bu notebook sadece **Transformer kısmını** eğitir; diğer modeller (lexicon, tf-idf, word2vec+FFN) kendi bilgisayarında CPU'da eğitilir.

Akış: kodu çek → veri → hızlı smoke test → tam eğitim (3 epoch) → ONNX FP32 export → INT8 quantization → Hugging Face Hub'a yükle.

**Runtime → Change runtime type → A100 GPU** seç.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## 1. Kodu al
GitHub reponun linkini yaz. (Repo private ise: Colab'a zip olarak yükleyip `!unzip` de yapabilirsin.)

In [2]:
REPO_URL = "https://github.com/akifkulll/LLM_Homework.git"
!git clone -q $REPO_URL /content/chat-moderation
%cd /content/chat-moderation
!pip install -q onnx onnxruntime onnxscript

/content/chat-moderation


## 2. Veri
OffensEval-TR 2020 indirilir; train resmi eğitim dosyasının %90'ı, val %10'u (tweet id hash'ine göre, her makinede aynı), test resmi test seti.

In [3]:
!python -m src.data

train:  28601 messages | offensive  5509 (19.3%)
val  :   3155 messages | offensive   622 (19.7%)
test :   3528 messages | offensive   716 (20.3%)


## 3. Smoke test (1-2 dakika)
1000 örnekle 1 epoch: kodun uçtan uca çalıştığını görmek için. Burada hata çıkarsa tam eğitime geçme, hatayı paylaş.

In [4]:
!python -m src.train_bert --quick

device=cuda  amp=torch.bfloat16
config.json: 100% 385/385 [00:00<00:00, 1.94MB/s]
tokenizer_config.json: 100% 60.0/60.0 [00:00<00:00, 340kB/s]
vocab.txt: 100% 251k/251k [00:00<00:00, 23.7MB/s]

model.safetensors: downloading bytes:  36% 160M/445M [00:01<00:01, 181MB/s, 12.9MB/s  ]
model.safetensors: downloading bytes:  53% 237M/445M [00:01<00:01, 184MB/s, 20.3MB/s  ]
model.safetensors: downloading bytes:  61% 271M/445M [00:02<00:00, 179MB/s, 23.6MB/s  ]
model.safetensors: reconstructing file:  27% 119M/445M [00:02<00:06, 50.6MB/s, 10.3MB/s  ]
model.safetensors: downloading bytes:  72% 319M/445M [00:02<00:00, 198MB/s, 26.0MB/s  ]
model.safetensors: downloading bytes:  87% 389M/445M [00:02<00:00, 212MB/s, 32.5MB/s  ]
model.safetensors: downloading bytes:  95% 421M/445M [00:02<00:00, 234MB/s, 34.3MB/s  ]
model.safetensors: reconstructing file:  81% 362M/445M [00:02<00:00, 339MB/s, 14.6MB/s  ] 
model.safetensors: downloading bytes: 100% 421M/421M [00:03<00:00, 131MB/s, 37.0MB/s  ]
model.sa

## 4. Tam eğitim
BERTurk + sınıflandırma başlığı, AdamW lr=2e-5, batch 32, 3 epoch, bf16. Her epoch sonunda val macro-F1 ölçülür, en iyi checkpoint ve karar eşiği (threshold) kaydedilir.

In [5]:
!python -m src.train_bert --epochs 3 --lr 2e-5 --batch 32

device=cuda  amp=torch.bfloat16
Loading weights: 100% 199/199 [00:00<00:00, 6968.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized b

## 5. ONNX export + INT8 quantization
- FP32 ONNX export, ardından PyTorch ile aynı çıktıyı verdiği kontrol edilir (assert)
- ONNX Runtime **dynamic quantization**: ağırlıklar INT8, aktivasyonlar çalışma anında quantize edilir
- INT8 dosyasının gerçekten küçüldüğü kontrol edilir, iki modelin de test macro-F1'i yazdırılır

Latency burada ölçülmez: nihai tablo kendi bilgisayarında `python -m src.benchmark` ile çıkar.

In [6]:
!python -m src.export_onnx
!ls -lh models/berturk_onnx

Loading weights: 100% 201/201 [00:00<00:00, 6438.39it/s]
[torch.onnx] Obtain model graph for `LogitsOnly([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LogitsOnly([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: batch will not be used, since it shares the same shape constraints with another axis: batch.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: seq will not be used, since it shares the same shape constraints with another axis: seq.
 

## 6. Hugging Face Hub'a yükle
Önce huggingface.co'da hesap aç ve **write** yetkili bir token oluştur. Aşağıdaki hücrede token'ı kendin gir. Sonra `HF_REPO` adını yaz.

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
HF_REPO = "akifkulll/berturk-offenseval-tr-onnx"  # <-- değiştir
!python scripts/upload_models.py --repo $HF_REPO

Found 5 files to upload
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  1.10MB · 2.20MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  9.24MB · 6.42MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  29.0MB · 16.3MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 5
  Preparing   ████████████████████  5 / 5 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  49.3MB · 23.6MB/s
  Co

In [9]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo("akifkulll/berturk-offenseval-tr", exist_ok=True)
api.upload_folder(folder_path="/content/chat-moderation/models/berturk_fp32_hf",
                  repo_id="akifkulll/berturk-offenseval-tr")

CommitInfo(commit_url='https://huggingface.co/akifkulll/berturk-offenseval-tr/commit/ec5f359dfa7dd18a0ca39f4719046859951f7e8d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='ec5f359dfa7dd18a0ca39f4719046859951f7e8d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/akifkulll/berturk-offenseval-tr', endpoint='https://huggingface.co', repo_type='model', repo_id='akifkulll/berturk-offenseval-tr'), pr_revision=None, pr_num=None)

### (Alternatif) Google Drive'a kopyala
HF kullanmak istemezsen klasörü Drive'a kopyala, bilgisayarında `models/berturk_onnx/` içine koy.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/chat-moderation && cp -r models/berturk_onnx /content/drive/MyDrive/chat-moderation/